# 16. Regla de señalamiento

Prueba de la regla que señala cuándo podría justificarse pedir el ajuste del piso (tarea 19 del backlog, HU-07). Una estación se señala si la banda superior del máximo pronosticado queda por debajo del piso: en ese caso, incluso el escenario alto de demanda queda por debajo de lo que la Cooperativa debe declarar.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from modelado import asignar_estaciones, calcular_maximos_estacionales
from regulacion import senalar_ajuste, calcular_piso

## 1. Prueba de la regla

Se prueban tres casos con el piso del verano 2026, de 39,82 MW: una banda superior por debajo del piso, una igual al piso y el pronóstico real del verano 2026. Solo el primero tiene que quedar señalado: si la banda superior alcanza el piso, el piso se mantiene como referencia de declaración. La regla informa además por cuánto supera el piso al pronóstico central y a la banda superior.

In [3]:
print(senalar_ajuste(38.0, 39.0, 39.82))
print(senalar_ajuste(38.0, 39.82, 39.82))
print(senalar_ajuste(45.06, 47.05, 39.82))

{'SENALADO': True, 'DIF_CENTRAL_MW': 1.82, 'DIF_SUPERIOR_MW': 0.82}
{'SENALADO': False, 'DIF_CENTRAL_MW': 1.82, 'DIF_SUPERIOR_MW': 0.0}
{'SENALADO': False, 'DIF_CENTRAL_MW': -5.24, 'DIF_SUPERIOR_MW': -7.23}


## 2. Aplicación a las estaciones pasadas

Se aplica la regla al pronóstico retrospectivo de cada estación, y se compara con lo que pasó: si el máximo real quedó efectivamente por debajo del piso.

In [4]:
retrospectivo = pd.read_csv(Path("../data/processed/RETROSPECTIVO_CTRELEUW.csv"))
serie = pd.read_csv(Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv"), parse_dates=["MES"])

In [5]:
serie = asignar_estaciones(serie)
maximos = calcular_maximos_estacionales(serie)
maximos = calcular_piso(maximos)
retrospectivo = retrospectivo.merge(maximos[["ESTACION", "ANIO_ESTACION", "PISO_MW"]], on=["ESTACION", "ANIO_ESTACION"], how="left")

In [6]:
senales = [senalar_ajuste(fila["Q50_MW"], fila["Q90_MW"], fila["PISO_MW"]) for _, fila in retrospectivo.iterrows()]
retrospectivo = pd.concat([retrospectivo, pd.DataFrame(senales)], axis=1)
retrospectivo["REAL_BAJO_PISO"] = retrospectivo["MAXIMO_MW"] < retrospectivo["PISO_MW"]
retrospectivo[["ESTACION", "ANIO_ESTACION", "MAXIMO_MW", "Q50_MW", "Q90_MW", "PISO_MW", "SENALADO", "DIF_SUPERIOR_MW", "REAL_BAJO_PISO"]]

,ESTACION,ANIO_ESTACION,MAXIMO_MW,Q50_MW,Q90_MW,PISO_MW,SENALADO,DIF_SUPERIOR_MW,REAL_BAJO_PISO
0,INVIERNO,2021,42.352,46.027273,47.260814,38.524,False,-8.737,False
1,INVIERNO,2022,44.156,45.302328,46.840297,38.117,False,-8.723,False
2,INVIERNO,2023,42.448,45.017425,46.953116,39.740,False,-7.213,False
3,INVIERNO,2024,41.960,44.313533,46.699444,38.203,False,-8.496,False
4,INVIERNO,2025,44.436,43.863804,46.760655,37.764,False,-8.997,False
5,VERANO,2021,44.380,44.340467,46.007473,39.956,False,-6.051,False
6,VERANO,2022,45.528,44.314921,45.871326,39.942,False,-5.929,False
7,VERANO,2023,48.340,44.551764,45.881774,40.975,False,-4.907,False
8,VERANO,2024,47.856,45.065222,46.901773,43.506,False,-3.396,False
9,VERANO,2025,44.244,45.233164,47.076460,35.892,False,-11.184,False


## Conclusión

- La regla se comporta como define la HU-07: señala solo cuando la banda superior queda estrictamente por debajo del piso.
- Ninguna de las 10 estaciones pasadas habría sido señalada, y en ninguna el máximo real quedó por debajo del piso. La regla no dio falsas alarmas, pero tampoco tuvo un caso en el que debiera señalar, así que su capacidad de detectarlo no pudo comprobarse con los datos disponibles.
- El verano 2026 tampoco queda señalado: la banda superior, de 47,05 MW, supera el piso en 7,23 MW.